# Model Training & Evaluating

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score,learning_curve,LearningCurveDisplay
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Lasso,Ridge,ElasticNet
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler,RobustScaler
from feature_engine.encoding import MeanEncoder,OneHotEncoder,RareLabelEncoder,CountFrequencyEncoder
from feature_engine.imputation import AddMissingIndicator,ArbitraryNumberImputer,CategoricalImputer
from feature_engine.selection import DropFeatures
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from optuna.visualization.matplotlib import plot_param_importances,plot_optimization_history,plot_timeline
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import seaborn as sns
import time
import joblib
import duckdb
from pathlib import Path
import sys
sys.path.append('../src')
from py_def_class import Add_Column,RFPermutationRegressorSelector,GroupTimeSplit,previous_stage_param_range,group_time_learning_curve,CastColumnsToObject,abandoned_lap,inspect_quali_time_distribution,f1_rule_era,get_sprint_session_name,build_session_paths,pick_quali_boundaries,practice_quali_new_pred,def_apply_offset_mean

pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv(r'..\data\processed\Cleaned_quali_f1.csv')
#dropping the track temperature columns
filter_out = set(fo for fo in df.columns if 'TrackTemp' in fo)
filter_out.add('Qualifying_Date')
df = df.drop(filter_out,axis=1)
df = df[df['laptime_sum_sectortimes_quali'].notna()].copy()
df.head()

In [ ]:
df.columns

In [ ]:
df[(df['Season'].eq(2026)) & (df['GP'].eq('Hungarian GP'))][['Driver','Team','GP','laptime_sum_sectortimes_quali']]

## Splitting into Train & Test

In [ ]:
target = "laptime_sum_sectortimes_quali"
cols_not_in_X = set(filter_out) | {target}
train_thresh = 2024
test_thresh = 2025

X_train = df[df['Season']<=train_thresh].drop([target,'LapTimeDiff_quali','Session'],axis=1)
X_train.to_csv('../data/model_data/train/X_train.csv',index=False)
X_test = df[df['Season'].eq(test_thresh)].drop([target,'LapTimeDiff_quali','Session'],axis=1)
X_test.to_csv('../data/model_data/test/X_test.csv',index=False)

#-------------- checking for any leak or logic errors in X features --------------
#checking if we do not have any overlapping indexies
assert set(X_train.index).isdisjoint(set(X_test.index)),"Having overlapping indexies in the X feature-set"
#checking if features that should be dropped are still in the train and/or test sets
assert set(X_train.columns).isdisjoint(cols_not_in_X),"Having dropped features are still in the Train X features"
assert set(X_test.columns).isdisjoint(cols_not_in_X),"Having dropped features are still in the Test X features"
assert X_train['Season'].max() < X_test['Season'].min(),"Max Season value of X train is not smaller than min X test Season value"


y_train = df[df['Season']<=train_thresh][target]
y_train.to_csv('../data/model_data/train/y_train.csv',index=False)
y_test = df[df['Season'].eq(test_thresh)][target]
y_test.to_csv('../data/model_data/test/y_test.csv',index=False)
#-------------- checking for any logic errors in y feature --------------
assert set(y_train.index).isdisjoint(set(y_test.index)),"Having overlapping indexies in the y feature-set"

## Basline Model - Dummy Regressor

In [ ]:
for s in ['mean','median']:
    DR = DummyRegressor(strategy=s)
    DR.fit(X_train,y_train)
    y_pred = DR.predict(X_test)

    MAE_DR = mean_absolute_error(y_test,y_pred)
    print(f"Dummy Regressor {s} - MAE: {MAE_DR}")

DR = DummyRegressor(strategy='median')
DR.fit(X_train,y_train)
y_pred = DR.predict(X_test)

MAE_DR = mean_absolute_error(y_test,y_pred)
joblib.dump(MAE_DR,'..\model\eval\dummy_regression_performance.joblib')

This will serve as the overall baseline score that all subsequent models must outperform. Next, we will establish a gradient-boosting-based benchmark model using a HistGradientBoosting Regressor, which will be hyperparameter tuned and used as a reference for evaluating LightGBM and XGBoost. We use HistGradientBoosting, because it can deal with features, which contain NaN values. As already established, not every weekend is a sprint-weekend, and therefore practice two and three do not contain always values, and the same for the sprint-weekend columns.

## Tree-Based-Baseline HistGradientBoosting
### Training the model

In [ ]:
def stage1(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',0.001,0.01,log=True)
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',0.001,0.03,log=True)
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',0.001,0.03,log=True)
    mean_smoothing = trial.suggest_float('mean_smoothing',0.01,10.0,log=True)
    feature_selector = trial.suggest_float('feature_selector',0.0001,0.1,log=True)

    max_iter = trial.suggest_int('max_iter',100,500)
    loss = trial.suggest_categorical('loss',['absolute_error','squared_error'])
    learning_rate = trial.suggest_float('learning_rate',0.02,0.1,log=True)
    max_depth = trial.suggest_int('max_depth',2,4)
    #min_samples_split = trial.suggest_int('min_samples_split',8,32)
    min_samples_leaf = trial.suggest_int('min_samples_leaf',15,50)
    max_leaf_nodes = None
    max_features = trial.suggest_float('max_features',0.6,0.9)
    max_bins = trial.suggest_int('max_bins',50,150)
    l2_regularization = trial.suggest_float('l2_regularization',0.01,10.0,log=True)
    #n_jobs=1

    hgb_param = {
        'max_iter':max_iter,
        'loss':loss,
        'learning_rate':learning_rate,
        'max_depth':max_depth,
        #'min_samples_split':min_samples_split,
        'min_samples_leaf':min_samples_leaf,
        'max_leaf_nodes':max_leaf_nodes,
        'max_features':max_features,
        'max_bins':max_bins,
        'l2_regularization':l2_regularization,
        "early_stopping": False,
        'random_state':101,
        #'n_jobs':n_jobs
    }

    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        #
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(
                col=mean_freq,
                name_= ['_mean','_frequency']
            )),
            ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RandomForest",HistGradientBoostingRegressor(**hgb_param))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study1=optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=3)
)

start_time = time.time()
study1.optimize(stage1,n_trials=40,n_jobs=-1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study1)
plt.title("Parameter Importance for Study 1")
plt.show()

plot_optimization_history(study1)
plt.title("Optimazation History for Study 1")
plt.show()

plot_timeline(study1)
plt.title("Timeline for Study 1")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_1_params = study1.best_params
study_1_best_trial = study1.best_value
study_1_run_time = end_time - start_time
print(f"Run time of study 1: {study_1_run_time}")

#### starting stage 2
def stage2(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[1])
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[1])
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[1])
    mean_smoothing = trial.suggest_float('mean_smoothing',previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[0],previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[1])
    feature_selector = trial.suggest_float('feature_selector',previous_stage_param_range(study_1_params['feature_selector'],0.3)[0],previous_stage_param_range(study_1_params['feature_selector'],0.3)[1])

    max_iter = trial.suggest_int('max_iter',previous_stage_param_range(study_1_params['max_iter'],0.3)[0],previous_stage_param_range(study_1_params['max_iter'],0.3)[1])
    loss = study_1_params['loss']
    learning_rate = trial.suggest_float('learning_rate',previous_stage_param_range(study_1_params['learning_rate'],0.3)[0],previous_stage_param_range(study_1_params['learning_rate'],0.3)[1])
    max_depth = trial.suggest_int('max_depth',previous_stage_param_range(study_1_params['max_depth'],0.3)[0],previous_stage_param_range(study_1_params['max_depth'],0.3)[1])
    #min_samples_split = trial.suggest_int('min_samples_split',8,32)
    min_samples_leaf = trial.suggest_int('min_samples_leaf',previous_stage_param_range(study_1_params['min_samples_leaf'],0.3)[0],previous_stage_param_range(study_1_params['min_samples_leaf'],0.3)[1])
    max_leaf_nodes = None
    max_features = trial.suggest_float('max_features',previous_stage_param_range(study_1_params['max_features'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['max_features'],0.3,[0.01,1.0])[1])
    max_bins = trial.suggest_int('max_bins',previous_stage_param_range(study_1_params['max_bins'],0.3)[0],previous_stage_param_range(study_1_params['max_bins'],0.3)[1])
    l2_regularization = trial.suggest_float('l2_regularization',previous_stage_param_range(study_1_params['l2_regularization'],0.3)[0],previous_stage_param_range(study_1_params['l2_regularization'],0.3)[1])
    #n_jobs=1

    hgb_param = {
        'max_iter':max_iter,
        'loss':loss,
        'learning_rate':learning_rate,
        'max_depth':max_depth,
        #'min_samples_split':min_samples_split,
        'min_samples_leaf':min_samples_leaf,
        'max_leaf_nodes':max_leaf_nodes,
        'max_features':max_features,
        'max_bins':max_bins,
        'l2_regularization':l2_regularization,
        "early_stopping": False,
        'random_state':101,
        #'n_jobs':n_jobs
    }
    if hgb_param['max_features'] > 1.0:
        hgb_param['max_features'] == 1.0
    elif hgb_param['max_features'] <0.0:
        hgb_param['max_features'] == 0.01

    #adding the fold_idx for the pruner, so the pruner knows, which trial we are
    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        #
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(
                col=mean_freq,
                name_= ['_mean','_frequency']
            )),
            ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RandomForest",HistGradientBoostingRegressor(**hgb_param))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study2 = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=2)
)

start_time = time.time()
study2.optimize(stage2,n_trials=60,n_jobs=1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study2)
plt.title("Parameter Importance for Study 2")
plt.show()

plot_optimization_history(study2)
plt.title("Optimization History for Study 2")
plt.show()

plot_timeline(study2)
plt.title("Timeline for Study 2")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_2_params = study2.best_params
study_2_params['loss'] = study_1_params['loss']
study_2_params['early_stopping'] = False
study_2_best_trial = study2.best_value
study_2_run_time = end_time - start_time
print(f"Run time of study 2: {study_2_run_time}")

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_param = study_1_params
else:
    print("Study 2 led to the best result")
    best_param = study_2_params

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_score = study_1_best_trial
else:
    print("Study 2 led to the best result")
    best_score = study_2_best_trial

total_time = study_1_run_time + study_2_run_time

joblib.dump(best_param,r'..\model\params\histgradientboosting_best_param.joblib')
joblib.dump(best_score,r'..\model\eval\histgradientboosting_best_score.joblib')
joblib.dump(total_time,r'..\model\run_time\histgradientboosting_run_time.joblib')

### Learning Curve

In [ ]:
best_param = joblib.load(r'..\model\params\histgradientboosting_best_param.joblib')

#list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
missing_in_sprint_we = [sprint_not for sprint_not in X_train.columns if 'p2' in sprint_not or 'p3' in sprint_not]
#
#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

cv=GroupTimeSplit(
    group_column='gp_id',
    n_splits=5,
    test_group_size=3
)

hgb_param = {
    'max_iter':best_param['max_iter'],
    'loss':best_param['loss'],
    'learning_rate':best_param['learning_rate'],
    'max_depth':best_param['max_depth'],
    #'min_samples_split':min_samples_split,
    'min_samples_leaf':best_param['min_samples_leaf'],
    'max_leaf_nodes':None,
    'max_features': best_param['max_features'],
    'max_bins':best_param['max_bins'],
    'l2_regularization':best_param['l2_regularization'],
    'early_stopping':False,
    'random_state':101,
    #'n_jobs':n_jobs
}

final_hist_pipeline = rf_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(
        col=mean_freq,
        name_= ['_mean','_frequency']
    )),
    ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("RandomForest",HistGradientBoostingRegressor(**hgb_param))
])

train_group_sizes, train_mae, val_mae = (
    group_time_learning_curve(
        estimator=final_hist_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        train_fractions=np.linspace(0.1, 1.0, 10),
    )
)

for size, fold_train_mae, fold_val_mae in zip(
    train_group_sizes,
    train_mae,
    val_mae,
):
    print(
        f"{size:.1f} GP groups | "
        f"Train MAE: {fold_train_mae.mean():.3f} "
        f"± {fold_train_mae.std():.3f} | "
        f"Validation MAE: {fold_val_mae.mean():.3f} "
        f"± {fold_val_mae.std():.3f} | "
        f"Gap: {fold_val_mae.mean() - fold_train_mae.mean():.3f}"
    )

plt.figure(figsize=(12,6))
display = LearningCurveDisplay(
    train_sizes=train_group_sizes,
    train_scores=train_mae,
    test_scores=val_mae,
    score_name="MAE",
)

display.plot()

## Light GBM
### Training the model

In [ ]:
def stage1(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',0.001,0.01,log=True)
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',0.001,0.03,log=True)
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',0.001,0.03,log=True)
    mean_smoothing = trial.suggest_float('mean_smoothing',0.01,10.0,log=True)
    feature_selector = trial.suggest_float('feature_selector',0.0001,0.1,log=True)

    objective = trial.suggest_categorical('objective',['regression','regression_l1','huber'])
    max_bin = trial.suggest_int('max_bin',56,225)
    num_iterations = trial.suggest_int('num_iterations',100,500)
    learning_rate = trial.suggest_float('learning_rate',0.02,0.1,log=True)
    num_leaves = trial.suggest_int('num_leaves',16,64)
    max_depth = trial.suggest_int('max_depth',2,4)
    min_data_in_leaf = trial.suggest_int('min_samples_leaf',5,20)
    min_sum_hessian_in_leaf = trial.suggest_float('min_sum_hessian_in_leaf',0.001,0.1,log=True)
    bagging_fraction = trial.suggest_float('bagging_fraction',0.1,0.9)
    bagging_freq = trial.suggest_int('bagging_freq',1,5)
    colsample_bytree = trial.suggest_float('colsample_bytree',0.01,0.9)
    feature_fraction_bynode = trial.suggest_float('feature_fraction_bynode',0.01,0.9)
    extra_trees = trial.suggest_categorical('extra_trees',[True,False])
    max_delta_step = trial.suggest_float('max_delta_step',0.01,0.9)
    reg_alpha = trial.suggest_float('reg_alpha',0.01,5.0,log=True)
    reg_lambda = trial.suggest_float('reg_lambda',0.01,5.0,log=True)

    lightgbm_params = {
        'objective':objective,
        'max_bin':max_bin,
        'num_iterations':num_iterations,
        'learning_rate':learning_rate,
        'num_leaves':num_leaves,
        'num_threads':1,
        'max_depth':max_depth,
        'min_data_in_leaf':min_data_in_leaf,
        'min_sum_hessian_in_leaf':min_sum_hessian_in_leaf,
        'bagging_fraction': bagging_fraction,
        'bagging_freq':bagging_freq,
        'colsample_bytree':colsample_bytree,
        'feature_fraction_bynode':feature_fraction_bynode,
        'extra_trees':extra_trees,
        'max_delta_step':max_delta_step,
        'reg_alpha':reg_alpha,
        'reg_lambda':reg_lambda,
        'bagging_seed':101,
        'feature_fraction_seed':101,
        'random_state':101,
        'extra_seed':101,
    }

    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        #
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(
                col=mean_freq,
                name_= ['_mean','_frequency']
            )),
            ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("LightGBM_Regressor",LGBMRegressor(**lightgbm_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study1=optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=3)
)

start_time = time.time()
study1.optimize(stage1,n_trials=40,n_jobs=-1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study1)
plt.title("Parameter Importance for Study 1")
plt.show()

plot_optimization_history(study1)
plt.title("Optimazation History for Study 1")
plt.show()

plot_timeline(study1)
plt.title("Timeline for Study 1")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_1_params = study1.best_params
study_1_best_trial = study1.best_value
study_1_run_time = end_time - start_time
print(f"Run time of study 1: {study_1_run_time}")

#### starting stage 2
def stage2(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[1])
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[1])
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[1])
    mean_smoothing = trial.suggest_float('mean_smoothing',previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[0],previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[1])
    feature_selector = trial.suggest_float('feature_selector',previous_stage_param_range(study_1_params['feature_selector'],0.3)[0],previous_stage_param_range(study_1_params['feature_selector'],0.3)[1])

    objective = study_1_params['objective']
    max_bin = trial.suggest_int('max_bin',previous_stage_param_range(study_1_params['max_bin'],0.3)[0],previous_stage_param_range(study_1_params['max_bin'],0.3)[1])
    num_iterations = trial.suggest_int('num_iterations',previous_stage_param_range(study_1_params['num_iterations'],0.3)[0],previous_stage_param_range(study_1_params['num_iterations'],0.3)[1])
    learning_rate = trial.suggest_float('learning_rate',previous_stage_param_range(study_1_params['learning_rate'],0.3)[0],previous_stage_param_range(study_1_params['learning_rate'],0.3)[1])
    num_leaves = trial.suggest_int('num_leaves',previous_stage_param_range(study_1_params['num_leaves'],0.3)[0],previous_stage_param_range(study_1_params['num_leaves'],0.3)[1])
    max_depth = trial.suggest_int('max_depth',previous_stage_param_range(study_1_params['max_depth'],0.3)[0],previous_stage_param_range(study_1_params['max_depth'],0.3)[1])
    min_data_in_leaf = trial.suggest_int('min_samples_leaf',previous_stage_param_range(study_1_params['min_samples_leaf'],0.3)[0],previous_stage_param_range(study_1_params['min_samples_leaf'],0.3)[1])
    min_sum_hessian_in_leaf = trial.suggest_float('min_sum_hessian_in_leaf',previous_stage_param_range(study_1_params['min_sum_hessian_in_leaf'],0.3)[0],previous_stage_param_range(study_1_params['min_sum_hessian_in_leaf'],0.3)[1])
    bagging_fraction = trial.suggest_float('bagging_fraction',previous_stage_param_range(study_1_params['bagging_fraction'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['bagging_fraction'],0.3,[0.01,1.0])[1])
    bagging_freq = trial.suggest_int('bagging_freq',previous_stage_param_range(study_1_params['bagging_freq'],0.3,)[0],previous_stage_param_range(study_1_params['bagging_freq'],0.3)[1])
    colsample_bytree = trial.suggest_float('colsample_bytree',previous_stage_param_range(study_1_params['colsample_bytree'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['colsample_bytree'],0.3,[0.01,1.0])[1])
    feature_fraction_bynode = trial.suggest_float('feature_fraction_bynode',previous_stage_param_range(study_1_params['feature_fraction_bynode'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['feature_fraction_bynode'],0.3,[0.01,1.0])[1])
    extra_trees = study_1_params['extra_trees']
    max_delta_step = trial.suggest_float('max_delta_step',previous_stage_param_range(study_1_params['max_delta_step'],0.3)[0],previous_stage_param_range(study_1_params['max_delta_step'],0.3)[1])
    reg_alpha = trial.suggest_float('reg_alpha',previous_stage_param_range(study_1_params['reg_alpha'],0.3)[0],previous_stage_param_range(study_1_params['reg_alpha'],0.3)[1])
    reg_lambda = trial.suggest_float('reg_lambda',previous_stage_param_range(study_1_params['reg_lambda'],0.3)[0],previous_stage_param_range(study_1_params['reg_lambda'],0.3)[1])

    lightgbm_params = {
        'objective':objective,
        'max_bin':max_bin,
        'num_iterations':num_iterations,
        'learning_rate':learning_rate,
        'num_leaves':num_leaves,
        'num_threads':1,
        'max_depth':max_depth,
        'min_data_in_leaf':min_data_in_leaf,
        'min_sum_hessian_in_leaf':min_sum_hessian_in_leaf,
        'bagging_fraction': bagging_fraction,
        'bagging_freq':bagging_freq,
        'colsample_bytree':colsample_bytree,
        'feature_fraction_bynode':feature_fraction_bynode,
        'extra_trees':extra_trees,
        'max_delta_step':max_delta_step,
        'reg_alpha':reg_alpha,
        'reg_lambda':reg_lambda,
        'bagging_seed':101,
        'feature_fraction_seed':101,
        'random_state':101,
        'extra_seed':101,
    }

    #adding the fold_idx for the pruner, so the pruner knows, which trial we are
    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        #
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(
                col=mean_freq,
                name_= ['_mean','_frequency']
            )),
            ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("LightGBM_Regressor",LGBMRegressor(**lightgbm_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study2 = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=2)
)

start_time = time.time()
study2.optimize(stage2,n_trials=60,n_jobs=1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study2)
plt.title("Parameter Importance for Study 2")
plt.show()

plot_optimization_history(study2)
plt.title("Optimization History for Study 2")
plt.show()

plot_timeline(study2)
plt.title("Timeline for Study 2")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_2_params = study2.best_params
study_2_params['objective'] = study_1_params['objective']
study_2_params['extra_trees'] = study_1_params['extra_trees']
study_2_best_trial = study2.best_value
study_2_run_time = end_time - start_time
print(f"Run time of study 2: {study_2_run_time}")

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_param = study_1_params
else:
    print("Study 2 led to the best result")
    best_param = study_2_params

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_score = study_1_best_trial
else:
    print("Study 2 led to the best result")
    best_score = study_2_best_trial

total_time = study_1_run_time + study_2_run_time

joblib.dump(best_param,r'..\model\params\lightgbm_best_param.joblib')
joblib.dump(best_score,r'..\model\eval\lightgbm_best_score.joblib')
joblib.dump(total_time,r'..\model\run_time\lightgbm_run_time.joblib')

### Learning Curve

In [ ]:
best_param = joblib.load(r'..\model\params\lightgbm_best_param.joblib')

#list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
missing_in_sprint_we = [sprint_not for sprint_not in X_train.columns if 'p2' in sprint_not or 'p3' in sprint_not]
#
#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

cv=GroupTimeSplit(
    group_column='gp_id',
    n_splits=5,
    test_group_size=3
)

lightgbm_params = {
        'objective':best_param['objective'],
        'max_bin':best_param['max_bin'],
        'num_iterations':best_param['num_iterations'],
        'learning_rate':best_param['learning_rate'],
        'num_leaves':best_param['num_leaves'],
        'num_threads':-1,
        'max_depth':best_param['max_depth'],
        'min_data_in_leaf':best_param['min_samples_leaf'],
        'min_sum_hessian_in_leaf':best_param['min_sum_hessian_in_leaf'],
        'bagging_fraction': best_param['bagging_fraction'],
        'bagging_freq': best_param['bagging_freq'],
        'colsample_bytree':best_param['colsample_bytree'],
        'feature_fraction_bynode':best_param['feature_fraction_bynode'],
        'extra_trees':best_param['extra_trees'],
        'max_delta_step':best_param['max_delta_step'],
        'reg_alpha':best_param['reg_alpha'],
        'reg_lambda':best_param['reg_lambda'],
        'bagging_seed':101,
        'feature_fraction_seed':101,
        'random_state':101,
        'extra_seed':101,
}

final_hist_pipeline = rf_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(
        col=mean_freq,
        name_= ['_mean','_frequency']
    )),
    ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("LightGBM_Regressor",LGBMRegressor(**lightgbm_params))
])

train_group_sizes, train_mae, val_mae = (
    group_time_learning_curve(
        estimator=final_hist_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        train_fractions=np.linspace(0.1, 1.0, 10),
    )
)

for size, fold_train_mae, fold_val_mae in zip(
    train_group_sizes,
    train_mae,
    val_mae,
):
    print(
        f"{size:.1f} GP groups | "
        f"Train MAE: {fold_train_mae.mean():.3f} "
        f"± {fold_train_mae.std():.3f} | "
        f"Validation MAE: {fold_val_mae.mean():.3f} "
        f"± {fold_val_mae.std():.3f} | "
        f"Gap: {fold_val_mae.mean() - fold_train_mae.mean():.3f}"
    )

plt.figure(figsize=(12,6))
display = LearningCurveDisplay(
    train_sizes=train_group_sizes,
    train_scores=train_mae,
    test_scores=val_mae,
    score_name="MAE",
)

display.plot()

## XGBoost
### Training the model

In [ ]:
def stage1(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',0.001,0.01,log=True)
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',0.001,0.03,log=True)
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',0.001,0.03,log=True)
    mean_smoothing = trial.suggest_float('mean_smoothing',0.01,10.0,log=True)
    feature_selector = trial.suggest_float('feature_selector',0.0001,0.1,log=True)

    objective = trial.suggest_categorical('objective',['reg:squarederror','reg:absoluteerror'])
    max_depth = trial.suggest_int('max_depth',2,4)
    n_estimators = trial.suggest_int('n_estimators',100,500)
    eta = trial.suggest_float('eta',0.001,0.2,log=True)
    subsample  = trial.suggest_float('subsample',0.01,1.0)
    min_child_weight = trial.suggest_float('min_child_weight',0.1,5.0,log=True)
    max_delta_step = trial.suggest_int('max_delta_step',1,6)
    colsample_bytree = trial.suggest_float('colsample_bytree',0.01,0.9)
    colsample_bylevel = trial.suggest_float('colsample_bylevel',0.01,0.9)
    colsample_bynode = trial.suggest_float('colsample_bynode',0.01,0.9)
    reg_alpha = trial.suggest_float('reg_alpha',0.01,5.0,log=True)
    reg_lambda = trial.suggest_float('reg_lambda',0.01,5.0,log=True)
    #max_leaves = trial.suggest_int('max_leaves',16,48)
    tree_method = trial.suggest_categorical('tree_method',['hist','approx'])
    max_bin = trial.suggest_int('max_bin',32,96)

    xgboost_params = {
        'objective':objective,
        'eval_metric':'mae',
        'max_depth':max_depth,
        'n_estimators':n_estimators,
        'eta':eta,
        'subsample':subsample,
        'min_child_weight':min_child_weight,
        'max_delta_step':max_delta_step,
        'colsample_bytree':colsample_bytree,
        'colsample_bylevel':colsample_bylevel,
        'colsample_bynode':colsample_bynode,
        'reg_alpha':reg_alpha,
        'reg_lambda':reg_lambda,
        'tree_method':tree_method,
        'max_bin':max_bin,
        'random_state':101,
        'n_jobs':1,
        'verbosity':0
        #'max_leaves':max_leaves
    }

    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        #
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(
                col=mean_freq,
                name_= ['_mean','_frequency']
            )),
            ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("XGBoost_Reg",XGBRegressor(**xgboost_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study1=optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=3)
)

start_time = time.time()
study1.optimize(stage1,n_trials=40,n_jobs=-1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study1)
plt.title("Parameter Importance for Study 1")
plt.show()

plot_optimization_history(study1)
plt.title("Optimazation History for Study 1")
plt.show()

plot_timeline(study1)
plt.title("Timeline for Study 1")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_1_params = study1.best_params
study_1_best_trial = study1.best_value
study_1_run_time = end_time - start_time
print(f"Run time of study 1: {study_1_run_time}")

#### starting stage 2
def stage2(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[1])
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[1])
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[1])
    mean_smoothing = trial.suggest_float('mean_smoothing',previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[0],previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[1])
    feature_selector = trial.suggest_float('feature_selector',previous_stage_param_range(study_1_params['feature_selector'],0.3)[0],previous_stage_param_range(study_1_params['feature_selector'],0.3)[1])

    objective = study_1_params['objective']
    max_depth = trial.suggest_int('max_depth',previous_stage_param_range(study_1_params['max_depth'],0.3)[0],previous_stage_param_range(study_1_params['max_depth'],0.3)[1])
    n_estimators = trial.suggest_int('n_estimators',previous_stage_param_range(study_1_params['n_estimators'],0.3)[0],previous_stage_param_range(study_1_params['n_estimators'],0.3)[1])
    eta = trial.suggest_float('eta',previous_stage_param_range(study_1_params['eta'],0.3)[0],previous_stage_param_range(study_1_params['eta'],0.3)[1])
    subsample = trial.suggest_float('subsample',previous_stage_param_range(study_1_params['subsample'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['subsample'],0.3,[0.01,1.0])[1])
    min_child_weight = trial.suggest_float('min_child_weight',previous_stage_param_range(study_1_params['min_child_weight'],0.3)[0],previous_stage_param_range(study_1_params['min_child_weight'],0.3)[1])
    max_delta_step = trial.suggest_int('max_delta_step',previous_stage_param_range(study_1_params['max_delta_step'],0.3)[0],previous_stage_param_range(study_1_params['max_delta_step'],0.3)[1])
    colsample_bytree = trial.suggest_float('colsample_bytree',previous_stage_param_range(study_1_params['colsample_bytree'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['colsample_bytree'],0.3,[0.01,1.0])[1])
    colsample_bylevel = trial.suggest_float('colsample_bylevel',previous_stage_param_range(study_1_params['colsample_bylevel'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['colsample_bylevel'],0.3,[0.01,1.0])[1])
    colsample_bynode = trial.suggest_float('colsample_bynode',previous_stage_param_range(study_1_params['colsample_bynode'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['colsample_bynode'],0.3,[0.01,1.0])[1])
    reg_alpha = trial.suggest_float('reg_alpha',previous_stage_param_range(study_1_params['reg_alpha'],0.3)[0],previous_stage_param_range(study_1_params['reg_alpha'],0.3)[1])
    reg_lambda = trial.suggest_float('reg_lambda',previous_stage_param_range(study_1_params['reg_lambda'],0.3)[0],previous_stage_param_range(study_1_params['reg_lambda'],0.3)[1])
    #max_leaves = trial.suggest_int('max_leaves',16,48)
    tree_method = study_1_params['tree_method']
    max_bin = trial.suggest_int('max_bin',previous_stage_param_range(study_1_params['max_bin'],0.3)[0],previous_stage_param_range(study_1_params['max_bin'],0.3)[1])

    xgboost_params = {
        'objective':objective,
        'eval_metric':'mae',
        'max_depth':max_depth,
        'n_estimators':n_estimators,
        'eta':eta,
        'subsample':subsample,
        'min_child_weight':min_child_weight,
        'max_delta_step':max_delta_step,
        'colsample_bytree':colsample_bytree,
        'colsample_bylevel':colsample_bylevel,
        'colsample_bynode':colsample_bynode,
        'reg_alpha':reg_alpha,
        'reg_lambda':reg_lambda,
        'tree_method':tree_method,
        'max_bin':max_bin,
        'random_state':101,
        'n_jobs':-1,
        'verbosity':0
        #'max_leaves':max_leaves
    }

    #adding the fold_idx for the pruner, so the pruner knows, which trial we are
    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        #
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(
                col=mean_freq,
                name_= ['_mean','_frequency']
            )),
            ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("XGBoost_Reg",XGBRegressor(**xgboost_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study2 = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=2)
)

start_time = time.time()
study2.optimize(stage2,n_trials=60,n_jobs=1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study2)
plt.title("Parameter Importance for Study 2")
plt.show()

plot_optimization_history(study2)
plt.title("Optimization History for Study 2")
plt.show()

plot_timeline(study2)
plt.title("Timeline for Study 2")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_2_params = study2.best_params
study_2_params['objective'] = study_1_params['objective']
study_2_params['tree_method'] = study_1_params['tree_method']
study_2_best_trial = study2.best_value
study_2_run_time = end_time - start_time
print(f"Run time of study 2: {study_2_run_time}")

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_param = study_1_params
    best_score = study_1_best_trial
else:
    print("Study 2 led to the best result")
    best_param = study_2_params
    best_score = study_2_best_trial

total_time = study_1_run_time + study_2_run_time

joblib.dump(best_param,r'..\model\params\xgboost_best_param.joblib')
joblib.dump(best_score,r'..\model\eval\xgboost_best_score.joblib')
joblib.dump(total_time,r'..\model\run_time\xgboost_run_time.joblib')

### Learning Curve

In [ ]:
best_param = joblib.load(r'..\model\params\xgboost_best_param.joblib')

#list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
missing_in_sprint_we = [sprint_not for sprint_not in X_train.columns if 'p2' in sprint_not or 'p3' in sprint_not]
#
#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

cv=GroupTimeSplit(
    group_column='gp_id',
    n_splits=5,
    test_group_size=3
)

xgboost_params = {
        'objective':best_param['objective'],
        'eval_metric':'mae',
        'max_depth':best_param['max_depth'],
        'n_estimators':best_param['n_estimators'],
        'eta':best_param['eta'],
        'subsample':best_param['subsample'],
        'min_child_weight':best_param['min_child_weight'],
        'max_delta_step':best_param['max_delta_step'],
        'colsample_bytree':best_param['colsample_bytree'],
        'colsample_bylevel':best_param['colsample_bylevel'],
        'colsample_bynode':best_param['colsample_bynode'],
        'reg_alpha':best_param['reg_alpha'],
        'reg_lambda':best_param['reg_lambda'],
        'tree_method':best_param['tree_method'],
        'max_bin':best_param['max_bin'],
        'random_state':101,
        'n_jobs':-1,
        'verbosity':0
        #'max_leaves':max_leaves
    }

final_hist_pipeline = rf_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(
        col=mean_freq,
        name_= ['_mean','_frequency']
    )),
    ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("XGBoost_Reg",XGBRegressor(**xgboost_params))
])

train_group_sizes, train_mae, val_mae = (
    group_time_learning_curve(
        estimator=final_hist_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        train_fractions=np.linspace(0.1, 1.0, 10),
    )
)

for size, fold_train_mae, fold_val_mae in zip(
    train_group_sizes,
    train_mae,
    val_mae,
):
    print(
        f"{size:.1f} GP groups | "
        f"Train MAE: {fold_train_mae.mean():.3f} "
        f"± {fold_train_mae.std():.3f} | "
        f"Validation MAE: {fold_val_mae.mean():.3f} "
        f"± {fold_val_mae.std():.3f} | "
        f"Gap: {fold_val_mae.mean() - fold_train_mae.mean():.3f}"
    )

plt.figure(figsize=(12,6))
display = LearningCurveDisplay(
    train_sizes=train_group_sizes,
    train_scores=train_mae,
    test_scores=val_mae,
    score_name="MAE",
)

display.plot()

## MLPRegressor
### Training the Model

In [ ]:
def stage1(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',0.001,0.01,log=True)
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',0.001,0.03,log=True)
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',0.001,0.03,log=True)
    mean_smoothing = trial.suggest_float('mean_smoothing',0.01,10.0,log=True)
    feature_selector = trial.suggest_float('feature_selector',0.0001,0.1,log=True)

    hidden_layer_sizes = trial.suggest_categorical('hidden_layer_sizes',[(8,16),(32,16),(64,32)])
    activation = trial.suggest_categorical('activation',["relu", "tanh"])
    solver = trial.suggest_categorical('solver',["adam", "lbfgs"])
    alpha = trial.suggest_float('alpha',0.0001,0.9)
    learning_rate_init = trial.suggest_float('learning_rate_init',0.0001,0.9)
    early_stopping = True
    validation_fraction = trial.suggest_float('validation_fraction',0.1,0.3)
    epsilon = trial.suggest_float('epsilon',0.00000001,0.01)
    max_iter = trial.suggest_int('max_iter',250,750)
    n_iter_no_change = trial.suggest_int('n_iter_no_change',2,10)
    

    mlp_params = {
        'hidden_layer_sizes':hidden_layer_sizes,
        'activation':activation,
        'solver':solver,
        'alpha':alpha,
        'learning_rate_init':learning_rate_init,
        'early_stopping':early_stopping,
        'validation_fraction':validation_fraction,
        'epsilon':epsilon,
        'max_iter':max_iter,
        'n_iter_no_change':n_iter_no_change,
        'random_state':101
    }

    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("MLP_Reg",MLPRegressor(**mlp_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study1=optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=3)
)

start_time = time.time()
study1.optimize(stage1,n_trials=40,n_jobs=-1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study1)
plt.title("Parameter Importance for Study 1")
plt.show()

plot_optimization_history(study1)
plt.title("Optimazation History for Study 1")
plt.show()

plot_timeline(study1)
plt.title("Timeline for Study 1")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_1_params = study1.best_params
study_1_params['early_stopping'] = True

study_1_best_trial = study1.best_value
study_1_run_time = end_time - start_time
print(f"Run time of study 1: {study_1_run_time}")

#### starting stage 2
def stage2(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[1])
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[1])
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[1])
    mean_smoothing = trial.suggest_float('mean_smoothing',previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[0],previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[1])
    feature_selector = trial.suggest_float('feature_selector',previous_stage_param_range(study_1_params['feature_selector'],0.3)[0],previous_stage_param_range(study_1_params['feature_selector'],0.3)[1])

    hidden_layer_sizes = study_1_params['hidden_layer_sizes']
    activation = study_1_params['activation']
    solver = study_1_params['solver']
    alpha = trial.suggest_float('alpha',previous_stage_param_range(study_1_params['alpha'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['alpha'],0.3,[0.01,1.0])[1])
    learning_rate_init = trial.suggest_float('learning_rate_init',previous_stage_param_range(study_1_params['learning_rate_init'],0.3)[0],previous_stage_param_range(study_1_params['learning_rate_init'],0.3)[1])
    early_stopping = study_1_params['early_stopping']
    validation_fraction = trial.suggest_float('validation_fraction',previous_stage_param_range(study_1_params['validation_fraction'],0.3,[0.01,1.0])[0],previous_stage_param_range(study_1_params['validation_fraction'],0.3,[0.01,1.0])[1])
    epsilon = trial.suggest_float('epsilon',previous_stage_param_range(study_1_params['epsilon'],0.3)[0],previous_stage_param_range(study_1_params['epsilon'],0.3)[1])
    max_iter = trial.suggest_int('max_iter',previous_stage_param_range(study_1_params['max_iter'],0.3)[0],previous_stage_param_range(study_1_params['max_iter'],0.3)[1])
    n_iter_no_change = trial.suggest_int('n_iter_no_change',previous_stage_param_range(study_1_params['n_iter_no_change'],0.3)[0],previous_stage_param_range(study_1_params['n_iter_no_change'],0.3)[1])
    
    mlp_params = {
        'hidden_layer_sizes':hidden_layer_sizes,
        'activation':activation,
        'solver':solver,
        'alpha':alpha,
        'learning_rate_init':learning_rate_init,
        'early_stopping':early_stopping,
        'validation_fraction':validation_fraction,
        'epsilon':epsilon,
        'max_iter':max_iter,
        'n_iter_no_change':n_iter_no_change,
        'random_state':101
    }

    #adding the fold_idx for the pruner, so the pruner knows, which trial we are
    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("MLP_Reg",MLPRegressor(**mlp_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study2 = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=2)
)

start_time = time.time()
study2.optimize(stage2,n_trials=60,n_jobs=1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study2)
plt.title("Parameter Importance for Study 2")
plt.show()

plot_optimization_history(study2)
plt.title("Optimization History for Study 2")
plt.show()

plot_timeline(study2)
plt.title("Timeline for Study 2")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_2_params = study2.best_params
study_2_params['hidden_layer_sizes'] = study_1_params['hidden_layer_sizes']
study_2_params['activation'] = study_1_params['activation']
study_2_params['solver'] = study_1_params['solver']
study_2_params['early_stopping'] = study_1_params['early_stopping']
study_2_best_trial = study2.best_value
study_2_run_time = end_time - start_time
print(f"Run time of study 2: {study_2_run_time}")

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_param = study_1_params
    best_score = study_1_best_trial
else:
    print("Study 2 led to the best result")
    best_param = study_2_params
    best_score = study_2_best_trial

total_time = study_1_run_time + study_2_run_time

joblib.dump(best_param,r'..\model\params\mlp_best_param.joblib')
joblib.dump(best_score,r'..\model\eval\mlp_best_score.joblib')
joblib.dump(total_time,r'..\model\run_time\mlp_run_time.joblib')

### Learning Curve

In [ ]:
best_param = joblib.load(r'..\model\params\mlp_best_param.joblib')

missing_input_features = [ip for ip in X_train.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
df_missing = X_train[missing_input_features]
missing_num = list(df_missing.select_dtypes(include=['number']).columns)
missing_ob = list(df_missing.select_dtypes(include=['object','category']).columns)

#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

cv=GroupTimeSplit(
    group_column='gp_id',
    n_splits=5,
    test_group_size=3
)

mlp_params = {
    'hidden_layer_sizes':best_param['hidden_layer_sizes'],
    'activation':best_param['activation'],
    'solver':best_param['solver'],
    'alpha':best_param['alpha'],
    'learning_rate_init':best_param['learning_rate_init'],
    'early_stopping':best_param['early_stopping'],
    'validation_fraction':best_param['validation_fraction'],
    'epsilon':best_param['epsilon'],
    'max_iter':best_param['max_iter'],
    'n_iter_no_change':best_param['n_iter_no_change'],
    'random_state':101
}

final_hist_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
    ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
    ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
    ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
    ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
    ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
    ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    #("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=0.0)),
    ("RobustScaler",RobustScaler()),
    ("MLP_Reg",MLPRegressor(**mlp_params))
])

train_group_sizes, train_mae, val_mae = (
    group_time_learning_curve(
        estimator=final_hist_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        train_fractions=np.linspace(0.1, 1.0, 10),
    )
)

for size, fold_train_mae, fold_val_mae in zip(
    train_group_sizes,
    train_mae,
    val_mae,
):
    print(
        f"{size:.1f} GP groups | "
        f"Train MAE: {fold_train_mae.mean():.3f} "
        f"± {fold_train_mae.std():.3f} | "
        f"Validation MAE: {fold_val_mae.mean():.3f} "
        f"± {fold_val_mae.std():.3f} | "
        f"Gap: {fold_val_mae.mean() - fold_train_mae.mean():.3f}"
    )

plt.figure(figsize=(12,6))
display = LearningCurveDisplay(
    train_sizes=train_group_sizes,
    train_scores=train_mae,
    test_scores=val_mae,
    score_name="MAE",
)

display.plot()

# Trying Linear Models

The most successful tree-based trials consistently selected a small set of features, usually when the feature-selection threshold was around `0.1`. These features were mainly the fastest driver lap times from the first, second and third free practice sessions, as well as the mean-encoded Grand Prix and qualifying tyre compound.

The exploratory analysis showed that the free-practice lap times have a strong approximately linear relationship with the qualifying lap time target. In addition, the previous models performed best with only a small number of highly predictive features. Therefore, linear regularized models will also be evaluated to check whether a simpler model can achieve similar or better generalization performance.

# Lasso Regression

Lasso Regression applies L1 regularization, which penalizes the absolute size of the model coefficients. This can shrink some coefficients exactly to zero, meaning that Lasso can also act as an embedded feature-selection method. Therefore, Lasso is useful when many linear features are available and only a subset of them is expected to be relevant.

However, when features are strongly correlated, Lasso may select only one feature from a correlated group and ignore the others. For this reason, Ridge Regression and Elastic Net will also be tested, as they are usually more stable when predictors are highly correlated.

## Training the Model

In [ ]:
def stage1(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',0.001,0.01,log=True)
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',0.001,0.03,log=True)
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',0.001,0.03,log=True)
    mean_smoothing = trial.suggest_float('mean_smoothing',0.01,10.0,log=True)
    feature_selector = trial.suggest_float('feature_selector',0.0001,0.1,log=True)

    alpha = trial.suggest_float('alpha',0.0001,0.01)
    max_iter = trial.suggest_int('max_iter',1000,5000)
    tol = trial.suggest_float('tol',0.0001,0.01)
    selection = trial.suggest_categorical('selection',['cyclic','random'])

    lasso_params = {
        'alpha':alpha,
        'max_iter':max_iter,
        'tol':tol,
        'selection':selection,
        'random_state':101
    }

    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        #missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("Lasso_Regressor",Lasso(**lasso_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study1=optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=3)
)

start_time = time.time()
study1.optimize(stage1,n_trials=40,n_jobs=-1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study1)
plt.title("Parameter Importance for Study 1")
plt.show()

plot_optimization_history(study1)
plt.title("Optimazation History for Study 1")
plt.show()

plot_timeline(study1)
plt.title("Timeline for Study 1")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_1_params = study1.best_params
study_1_best_trial = study1.best_value
study_1_run_time = end_time - start_time
print(f"Run time of study 1: {study_1_run_time}")

#### starting stage 2
def stage2(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[1])
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[1])
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[1])
    mean_smoothing = trial.suggest_float('mean_smoothing',previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[0],previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[1])
    feature_selector = trial.suggest_float('feature_selector',previous_stage_param_range(study_1_params['feature_selector'],0.3)[0],previous_stage_param_range(study_1_params['feature_selector'],0.3)[1])

    alpha = trial.suggest_float('alpha',previous_stage_param_range(study_1_params['alpha'],0.3,[0.001,1.0])[0],previous_stage_param_range(study_1_params['alpha'],0.3,[0.001,1.0])[1])
    max_iter = trial.suggest_int('max_iter',previous_stage_param_range(study_1_params['max_iter'],0.3)[0],previous_stage_param_range(study_1_params['max_iter'],0.3)[1])
    tol = trial.suggest_float('tol',previous_stage_param_range(study_1_params['tol'],0.3)[0],previous_stage_param_range(study_1_params['tol'],0.3)[1])
    selection = study_1_params['selection']

    lasso_params = {
        'alpha':alpha,
        'max_iter':max_iter,
        'tol':tol,
        'selection':selection,
        'random_state':101
    }

    #adding the fold_idx for the pruner, so the pruner knows, which trial we are
    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("Lasso_Regressor",Lasso(**lasso_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study2 = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=2)
)

start_time = time.time()
study2.optimize(stage2,n_trials=60,n_jobs=1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study2)
plt.title("Parameter Importance for Study 2")
plt.show()

plot_optimization_history(study2)
plt.title("Optimization History for Study 2")
plt.show()

plot_timeline(study2)
plt.title("Timeline for Study 2")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_2_params = study2.best_params
study_2_params['selection'] = study_1_params['selection']
study_2_best_trial = study2.best_value
study_2_run_time = end_time - start_time
print(f"Run time of study 2: {study_2_run_time}")

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_param = study_1_params
else:
    print("Study 2 led to the best result")
    best_param = study_2_params

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_score = study_1_best_trial
else:
    print("Study 2 led to the best result")
    best_score = study_2_best_trial

total_time = study_1_run_time + study_2_run_time

joblib.dump(best_param,r'..\model\params\lasso_best_param.joblib')
joblib.dump(best_score,r'..\model\eval\lasso_best_score.joblib')
joblib.dump(total_time,r'..\model\run_time\lasso_run_time.joblib')

## Learning Curve

In [ ]:
best_param = joblib.load(r'..\model\params\lasso_best_param.joblib')

missing_input_features = [ip for ip in X_train.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
df_missing = X_train[missing_input_features]
missing_num = list(df_missing.select_dtypes(include=['number']).columns)
missing_ob = list(df_missing.select_dtypes(include=['object','category']).columns)

#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

cv=GroupTimeSplit(
    group_column='gp_id',
    n_splits=5,
    test_group_size=3
)

lasso_params = {
        'alpha':best_param['alpha'],
        'max_iter':best_param['max_iter'],
        'tol':best_param['tol'],
        'selection':best_param['selection'],
        'random_state':101
    }

final_hist_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
    ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
    ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
    ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
    ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
    ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
    ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("RobustScaler",RobustScaler()),
    ("Lasso_Regressor",Lasso(**lasso_params))
])

train_group_sizes, train_mae, val_mae = (
    group_time_learning_curve(
        estimator=final_hist_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        train_fractions=np.linspace(0.1, 1.0, 10),
    )
)

for size, fold_train_mae, fold_val_mae in zip(
    train_group_sizes,
    train_mae,
    val_mae,
):
    print(
        f"{size:.1f} GP groups | "
        f"Train MAE: {fold_train_mae.mean():.3f} "
        f"± {fold_train_mae.std():.3f} | "
        f"Validation MAE: {fold_val_mae.mean():.3f} "
        f"± {fold_val_mae.std():.3f} | "
        f"Gap: {fold_val_mae.mean() - fold_train_mae.mean():.3f}"
    )

plt.figure(figsize=(12,6))
display = LearningCurveDisplay(
    train_sizes=train_group_sizes,
    train_scores=train_mae,
    test_scores=val_mae,
    score_name="MAE",
)

display.plot()

# Ridge Regression

## Training the Model

In [ ]:
def stage1(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',0.001,0.01,log=True)
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',0.001,0.03,log=True)
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',0.001,0.03,log=True)
    mean_smoothing = trial.suggest_float('mean_smoothing',0.01,10.0,log=True)
    feature_selector = trial.suggest_float('feature_selector',0.0001,0.1,log=True)

    alpha = trial.suggest_float('alpha',0.00001,0.1)
    max_iter = trial.suggest_int('max_iter',1000,5000)
    tol = trial.suggest_float('tol',0.0001,0.01)

    ridge_params = {
        'alpha':alpha,
        'max_iter':max_iter,
        'tol':tol,
        'random_state':101
    }

    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        #missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("Ridge_Regressor",Ridge(**ridge_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study1=optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=3)
)

start_time = time.time()
study1.optimize(stage1,n_trials=40,n_jobs=-1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study1)
plt.title("Parameter Importance for Study 1")
plt.show()

plot_optimization_history(study1)
plt.title("Optimazation History for Study 1")
plt.show()

plot_timeline(study1)
plt.title("Timeline for Study 1")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_1_params = study1.best_params
study_1_best_trial = study1.best_value
study_1_run_time = end_time - start_time
print(f"Run time of study 1: {study_1_run_time}")

#### starting stage 2
def stage2(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[1])
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[1])
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[1])
    mean_smoothing = trial.suggest_float('mean_smoothing',previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[0],previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[1])
    feature_selector = trial.suggest_float('feature_selector',previous_stage_param_range(study_1_params['feature_selector'],0.3)[0],previous_stage_param_range(study_1_params['feature_selector'],0.3)[1])

    alpha = trial.suggest_float('alpha',previous_stage_param_range(study_1_params['alpha'],0.3,[0.001,1.0])[0],previous_stage_param_range(study_1_params['alpha'],0.3,[0.001,1.0])[1])
    max_iter = trial.suggest_int('max_iter',previous_stage_param_range(study_1_params['max_iter'],0.3)[0],previous_stage_param_range(study_1_params['max_iter'],0.3)[1])
    tol = trial.suggest_float('tol',previous_stage_param_range(study_1_params['tol'],0.3)[0],previous_stage_param_range(study_1_params['tol'],0.3)[1])
    #selection = study_1_params['selection']

    ridge_params = {
        'alpha':alpha,
        'max_iter':max_iter,
        'tol':tol,
        'random_state':101
    }

    #adding the fold_idx for the pruner, so the pruner knows, which trial we are
    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        #missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("Ridge_Regressor",Ridge(**ridge_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study2 = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=2)
)

start_time = time.time()
study2.optimize(stage2,n_trials=60,n_jobs=1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study2)
plt.title("Parameter Importance for Study 2")
plt.show()

plot_optimization_history(study2)
plt.title("Optimization History for Study 2")
plt.show()

plot_timeline(study2)
plt.title("Timeline for Study 2")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_2_params = study2.best_params
#study_2_params['selection'] = study_1_params['selection']
study_2_best_trial = study2.best_value
study_2_run_time = end_time - start_time
print(f"Run time of study 2: {study_2_run_time}")

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_param = study_1_params
else:
    print("Study 2 led to the best result")
    best_param = study_2_params

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_score = study_1_best_trial
else:
    print("Study 2 led to the best result")
    best_score = study_2_best_trial

total_time = study_1_run_time + study_2_run_time

joblib.dump(best_param,r'..\model\params\ridge_best_param.joblib')
joblib.dump(best_score,r'..\model\eval\ridge_best_score.joblib')
joblib.dump(total_time,r'..\model\run_time\ridge_run_time.joblib')

## Learning Curve

In [ ]:
best_param = joblib.load(r'..\model\params\ridge_best_param.joblib')

missing_input_features = [ip for ip in X_train.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
df_missing = X_train[missing_input_features]
missing_num = list(df_missing.select_dtypes(include=['number']).columns)
missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)

#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))


cv=GroupTimeSplit(
    group_column='gp_id',
    n_splits=5,
    test_group_size=3
)

ridge_params = {
        'alpha':best_param['alpha'],
        'max_iter':best_param['max_iter'],
        'tol':best_param['tol'],
        'random_state':101
    }

final_hist_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
    ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
    ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
    ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
    ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
    ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
    ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("RobustScaler",RobustScaler()),
    ("Ridge_Regressor",Ridge(**ridge_params))
])

train_group_sizes, train_mae, val_mae = (
    group_time_learning_curve(
        estimator=final_hist_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        train_fractions=np.linspace(0.1, 1.0, 10),
    )
)

for size, fold_train_mae, fold_val_mae in zip(
    train_group_sizes,
    train_mae,
    val_mae,
):
    print(
        f"{size:.1f} GP groups | "
        f"Train MAE: {fold_train_mae.mean():.3f} "
        f"± {fold_train_mae.std():.3f} | "
        f"Validation MAE: {fold_val_mae.mean():.3f} "
        f"± {fold_val_mae.std():.3f} | "
        f"Gap: {fold_val_mae.mean() - fold_train_mae.mean():.3f}"
    )

plt.figure(figsize=(12,6))
display = LearningCurveDisplay(
    train_sizes=train_group_sizes,
    train_scores=train_mae,
    test_scores=val_mae,
    score_name="MAE",
)

display.plot()

# Elastic Net Regression
## Training the Model

In [ ]:
def stage1(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',0.001,0.01,log=True)
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',0.001,0.03,log=True)
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',0.001,0.03,log=True)
    mean_smoothing = trial.suggest_float('mean_smoothing',0.01,10.0,log=True)
    feature_selector = trial.suggest_float('feature_selector',0.0001,0.1,log=True)

    alpha = trial.suggest_float('alpha',0.00001,0.1)
    l1_ratio = trial.suggest_float('l1_ratio',0.001,0.5)
    max_iter = trial.suggest_int('max_iter',1000,5000)
    tol = trial.suggest_float('tol',0.00001,0.001)
    selection = trial.suggest_categorical('selection',['cyclic','random'])

    elastic_net_params = {
        'alpha':alpha,
        'l1_ratio':l1_ratio,
        'max_iter':max_iter,
        'tol':tol,
        'selection':selection,
        'random_state':101
    }

    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        #missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("Elastic_Net_Regressor",ElasticNet(**elastic_net_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study1=optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=3)
)

start_time = time.time()
study1.optimize(stage1,n_trials=40,n_jobs=-1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study1)
plt.title("Parameter Importance for Study 1")
plt.show()

plot_optimization_history(study1)
plt.title("Optimazation History for Study 1")
plt.show()

plot_timeline(study1)
plt.title("Timeline for Study 1")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_1_params = study1.best_params
study_1_best_trial = study1.best_value
study_1_run_time = end_time - start_time
print(f"Run time of study 1: {study_1_run_time}")

#### starting stage 2
def stage2(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[1])
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[1])
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[1])
    mean_smoothing = trial.suggest_float('mean_smoothing',previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[0],previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[1])
    feature_selector = trial.suggest_float('feature_selector',previous_stage_param_range(study_1_params['feature_selector'],0.3)[0],previous_stage_param_range(study_1_params['feature_selector'],0.3)[1])

    alpha = trial.suggest_float('alpha',previous_stage_param_range(study_1_params['alpha'],0.3,[0.001,1.0])[0],previous_stage_param_range(study_1_params['alpha'],0.3,[0.001,1.0])[1])
    l1_ratio = trial.suggest_float('l1_ratio',previous_stage_param_range(study_1_params['l1_ratio'],0.3,[0.001,1.0])[0],previous_stage_param_range(study_1_params['l1_ratio'],0.3,[0.001,1.0])[1])
    max_iter = trial.suggest_int('max_iter',previous_stage_param_range(study_1_params['max_iter'],0.3)[0],previous_stage_param_range(study_1_params['max_iter'],0.3)[1])
    tol = trial.suggest_float('tol',previous_stage_param_range(study_1_params['tol'],0.3)[0],previous_stage_param_range(study_1_params['tol'],0.3)[1])
    selection = study_1_params['selection']

    elastic_net_params = {
        'alpha':alpha,
        'l1_ratio':l1_ratio,
        'max_iter':max_iter,
        'tol':tol,
        'selection':selection,
        'random_state':101
    }

    #adding the fold_idx for the pruner, so the pruner knows, which trial we are
    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        #missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("Elastic_Net_Regressor",ElasticNet(**elastic_net_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study2 = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=2)
)

start_time = time.time()
study2.optimize(stage2,n_trials=60,n_jobs=1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study2)
plt.title("Parameter Importance for Study 2")
plt.show()

plot_optimization_history(study2)
plt.title("Optimization History for Study 2")
plt.show()

plot_timeline(study2)
plt.title("Timeline for Study 2")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_2_params = study2.best_params
study_2_params['selection'] = study_1_params['selection']
study_2_best_trial = study2.best_value
study_2_run_time = end_time - start_time
print(f"Run time of study 2: {study_2_run_time}")

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_param = study_1_params
else:
    print("Study 2 led to the best result")
    best_param = study_2_params

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_score = study_1_best_trial
else:
    print("Study 2 led to the best result")
    best_score = study_2_best_trial

total_time = study_1_run_time + study_2_run_time

joblib.dump(best_param,r'..\model\params\elastic_net_best_param.joblib')
joblib.dump(best_score,r'..\model\eval\elastic_net_best_score.joblib')
joblib.dump(total_time,r'..\model\run_time\elastic_net_run_time.joblib')

In [ ]:
#### starting stage 2
def stage2(trial):
    cv=GroupTimeSplit(
        group_column='gp_id',
        n_splits=5,
        test_group_size=3
    )
    
    mae_fold_lst = []

    tol_rare_label_driver = trial.suggest_float('tol_rare_label_driver',previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_driver'],0.3)[1])
    tol_rare_label_gp = trial.suggest_float('tol_rare_label_gp',previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_gp'],0.3)[1])
    tol_rare_label_compounds = trial.suggest_float('tol_rare_label_compounds',previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[0],previous_stage_param_range(study_1_params['tol_rare_label_compounds'],0.3)[1])
    mean_smoothing = trial.suggest_float('mean_smoothing',previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[0],previous_stage_param_range(study_1_params['mean_smoothing'],0.3)[1])
    feature_selector = trial.suggest_float('feature_selector',previous_stage_param_range(study_1_params['feature_selector'],0.3)[0],previous_stage_param_range(study_1_params['feature_selector'],0.3)[1])

    alpha = trial.suggest_float('alpha',previous_stage_param_range(study_1_params['alpha'],0.3,[0.001,1.0])[0],previous_stage_param_range(study_1_params['alpha'],0.3,[0.001,1.0])[1])
    l1_ratio = trial.suggest_float('l1_ratio',previous_stage_param_range(study_1_params['l1_ratio'],0.3,[0.001,1.0])[0],previous_stage_param_range(study_1_params['l1_ratio'],0.3,[0.001,1.0])[1])
    max_iter = trial.suggest_int('max_iter',previous_stage_param_range(study_1_params['max_iter'],0.3)[0],previous_stage_param_range(study_1_params['max_iter'],0.3)[1])
    tol = trial.suggest_float('tol',previous_stage_param_range(study_1_params['tol'],0.3)[0],previous_stage_param_range(study_1_params['tol'],0.3)[1])
    selection = study_1_params['selection']

    elastic_net_params = {
        'alpha':alpha,
        'l1_ratio':l1_ratio,
        'max_iter':max_iter,
        'tol':tol,
        'selection':selection,
        'random_state':101
    }

    #adding the fold_idx for the pruner, so the pruner knows, which trial we are
    for fold_idx, (train_idx,val_idx) in enumerate(cv.split(X_train,y_train)):
        X_train_,y_train_ = X_train.iloc[train_idx],y_train.iloc[train_idx]
        X_val,y_val = X_train.iloc[val_idx],y_train.iloc[val_idx]

        #list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
        #missing_in_sprint_we = [sprint_not for sprint_not in X_train_.columns if 'p2' in sprint_not or 'p3' in sprint_not]
        missing_input_features = [ip for ip in X_train_.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
        df_missing = X_train_[missing_input_features]
        missing_num = list(df_missing.select_dtypes(include=['number']).columns)
        missing_ob = list(df_missing.select_dtypes(exclude=['number']).columns)
        
        #mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        #'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
        mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
        'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
        mean_enc_cols = [m+'_mean' for m in mean_freq]
        freq_enc_cols = [f+'_frequency' for f in mean_freq]

        one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

        cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

        #tol => percentage/frequency of appearance
        rf_pipeline = Pipeline([
            ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=tol_rare_label_driver,replace_with='Rare_Driver')),
            ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=tol_rare_label_gp,replace_with='Rare_GP')),
            #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
            #because tree-based models can deal with NaN values
            ("RareLabelEncoder_Compounds",RareLabelEncoder(
                variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
                tol=tol_rare_label_compounds,
                n_categories=2,
                replace_with='Rare_Compound',
                missing_values='ignore'
            )),
            ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
            ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
            ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
            ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
            ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
            ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
            ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
            ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
            ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
            ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=mean_smoothing,unseen="encode")),
            ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
            ("Feature_Selector",RFPermutationRegressorSelector(threshold=feature_selector)),
            ("RobustScaler",RobustScaler()),
            ("Elastic_Net_Regressor",ElasticNet(**elastic_net_params))
        ]).fit(X_train_,y_train_)

        pred_val = rf_pipeline.predict(X_val)
        mae_fold = mean_absolute_error(y_val,pred_val)
        mae_fold_lst.append(mae_fold)
        running_mae = float(np.mean(mae_fold_lst))
        trial.report(running_mae, step=fold_idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
    mae_cv=np.mean(mae_fold_lst)
    return mae_cv

study2 = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=5,seed=101),
    pruner=MedianPruner(n_startup_trials=10,n_warmup_steps=2,n_min_trials=2)
)

start_time = time.time()
study2.optimize(stage2,n_trials=60,n_jobs=1,show_progress_bar=True)
plt.figure(figsize=(12,6))
plot_param_importances(study2)
plt.title("Parameter Importance for Study 2")
plt.show()

plot_optimization_history(study2)
plt.title("Optimization History for Study 2")
plt.show()

plot_timeline(study2)
plt.title("Timeline for Study 2")
plt.show()
end_time = time.time()

plt.rcdefaults()
plt.style.use("default")

study_2_params = study2.best_params
study_2_params['selection'] = study_1_params['selection']
study_2_best_trial = study2.best_value
study_2_run_time = end_time - start_time
print(f"Run time of study 2: {study_2_run_time}")

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_param = study_1_params
else:
    print("Study 2 led to the best result")
    best_param = study_2_params

if study_1_best_trial < study_2_best_trial:
    print("Study 1 led to the best result")
    best_score = study_1_best_trial
else:
    print("Study 2 led to the best result")
    best_score = study_2_best_trial

total_time = study_1_run_time + study_2_run_time

joblib.dump(best_param,r'..\model\params\elastic_net_best_param.joblib')
joblib.dump(best_score,r'..\model\eval\elastic_net_best_score.joblib')
joblib.dump(total_time,r'..\model\run_time\elastic_net_run_time.joblib')

## Learning Curve

In [ ]:
best_param = joblib.load(r'..\model\params\elastic_net_best_param.joblib')

missing_input_features = [ip for ip in X_train.columns if 'p1' in ip.lower() or 'p2' in ip.lower() or 'p3' in ip.lower() or 'sprint' in ip.lower()]
df_missing = X_train[missing_input_features]
missing_num = list(df_missing.select_dtypes(include=['number']).columns)
missing_ob = list(df_missing.select_dtypes(include=['object','category']).columns)

#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

one_hot_cols = ['Pace_profile','Type','Direction','Rule_Era']

cat_cols_to_cast = list(dict.fromkeys(missing_ob + mean_enc_cols + freq_enc_cols + one_hot_cols))

cv=GroupTimeSplit(
    group_column='gp_id',
    n_splits=5,
    test_group_size=3
)

elastic_net_params = {
        'alpha':best_param['alpha'],
        'l1_ratio':best_param['l1_ratio'],
        'max_iter':best_param['max_iter'],
        'tol':best_param['tol'],
        'selection':best_param['selection'],
        'random_state':101
    }

final_hist_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(col=mean_freq,name_= ['_mean','_frequency'])),
    ("Indicating_Missing_Value", AddMissingIndicator(missing_only=True,variables=missing_input_features)),
    ("NaN-Imputer_Numeric", ArbitraryNumberImputer(arbitrary_number=0,variables=missing_num)),
    ("Cast_Missing_Categorical_Before_Imputer", CastColumnsToObject(variables=missing_ob)),
    ("NaN-Imputer_Categorical", CategoricalImputer(imputation_method="missing",fill_value='-',variables=missing_ob)),
    ("Cast_Encoding_Cols_Before_Imputer", CastColumnsToObject(variables=mean_enc_cols + freq_enc_cols)),
    ("NaN_Imputer_Categorical_Encoding_Cols", CategoricalImputer(imputation_method="missing",fill_value="-",variables=mean_enc_cols + freq_enc_cols,)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=one_hot_cols)),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("RobustScaler",RobustScaler()),
    ("Elastic_Net_Regressor",ElasticNet(**elastic_net_params))
])

train_group_sizes, train_mae, val_mae = (
    group_time_learning_curve(
        estimator=final_hist_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        train_fractions=np.linspace(0.1, 1.0, 10),
    )
)

for size, fold_train_mae, fold_val_mae in zip(
    train_group_sizes,
    train_mae,
    val_mae,
):
    print(
        f"{size:.1f} GP groups | "
        f"Train MAE: {fold_train_mae.mean():.3f} "
        f"± {fold_train_mae.std():.3f} | "
        f"Validation MAE: {fold_val_mae.mean():.3f} "
        f"± {fold_val_mae.std():.3f} | "
        f"Gap: {fold_val_mae.mean() - fold_train_mae.mean():.3f}"
    )

plt.figure(figsize=(12,6))
display = LearningCurveDisplay(
    train_sizes=train_group_sizes,
    train_scores=train_mae,
    test_scores=val_mae,
    score_name="MAE",
)

display.plot()

# Performance of the models

In [ ]:
df_performance = pd.DataFrame()

#model_name
model_name = ['DummyRegressor','HistGradientBoosting','LightGBM','XGBoosting','MLP','Lasso','Ridge','Elastic Net']

#model scores
dummy_score = joblib.load(r'..\model\eval\dummy_regression_performance.joblib')
hist_score = joblib.load(r'..\model\eval\histgradientboosting_best_score.joblib')
lgbm_score = joblib.load(r'..\model\eval\lightgbm_best_score.joblib')
xgboost_score = joblib.load(r'..\model\eval\xgboost_best_score.joblib')
mlp_score = joblib.load(r'..\model\eval\mlp_best_score.joblib')
lasso_score = joblib.load(r'..\model\eval\lasso_best_score.joblib')
ridge_score = joblib.load(r'..\model\eval\ridge_best_score.joblib')
elastic_score = joblib.load(r'..\model\eval\elastic_net_best_score.joblib')
model_score_lst = [dummy_score,hist_score,lgbm_score,xgboost_score,mlp_score,lasso_score,ridge_score,elastic_score]

#run_time
hist_run_time = (joblib.load(r'..\model\run_time\histgradientboosting_run_time.joblib')/60)/60
lgbm_run_time = (joblib.load(r'..\model\run_time\lightgbm_run_time.joblib')/60)/60
xgboost_run_time = (joblib.load(r'..\model\run_time\xgboost_run_time.joblib')/60)/60
mlp_run_time = (joblib.load(r'..\model\run_time\mlp_run_time.joblib')/60)/60
lasso_runt_time = (joblib.load(r'..\model\run_time\lasso_run_time.joblib')/60)/60
ridge_runt_time = (joblib.load(r'..\model\run_time\ridge_run_time.joblib')/60)/60
elastic_runt_time = (joblib.load(r'..\model\run_time\elastic_net_run_time.joblib')/60)/60
model_run_time_lst = [((1/60)/60),hist_run_time,lgbm_run_time,xgboost_run_time,mlp_run_time,lasso_runt_time,ridge_runt_time,elastic_runt_time]

df_performance = pd.DataFrame(
    {
        'Models':model_name,
        'Model_Score':model_score_lst,
        'Model_Run_Time_in_h':model_run_time_lst
    }
).sort_values('Model_Score')
df_performance['score_rank'] = range(1,len(model_score_lst)+1)

sub_df = df_performance[['Models','Model_Run_Time_in_h']].sort_values('Model_Run_Time_in_h')
sub_df['run_time_rank'] = range(1,len(model_score_lst)+1)
df_performance = df_performance.merge(sub_df,how='inner',on=['Models','Model_Run_Time_in_h'])

df_performance

In [ ]:
best_prev_model = df_performance[df_performance['Model_Score'].eq(df_performance['Model_Score'].min())]
float(best_prev_model['Model_Score'].iloc[0])

After running multiple models, we come to the conclusion, that our best performing model is the HistGradientBoosting model. When it comes to the time efficen, HistGradientBoosting is rather at the lower end, but the socring performance is so good, that we will go with that.

## Final-Test Model

In [ ]:
best_param = joblib.load(r'..\model\params\lightgbm_best_param.joblib')

#list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
missing_in_sprint_we = [sprint_not for sprint_not in X_train.columns if 'p2' in sprint_not or 'p3' in sprint_not]
#
#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

lightgbm_params = {
        'objective':best_param['objective'],
        'max_bin':best_param['max_bin'],
        'num_iterations':best_param['num_iterations'],
        'learning_rate':best_param['learning_rate'],
        'num_leaves':best_param['num_leaves'],
        'num_threads':-1,
        'max_depth':best_param['max_depth'],
        'min_data_in_leaf':best_param['min_samples_leaf'],
        'min_sum_hessian_in_leaf':best_param['min_sum_hessian_in_leaf'],
        'bagging_fraction': best_param['bagging_fraction'],
        'bagging_freq': best_param['bagging_freq'],
        'colsample_bytree':best_param['colsample_bytree'],
        'feature_fraction_bynode':best_param['feature_fraction_bynode'],
        'extra_trees':best_param['extra_trees'],
        'max_delta_step':best_param['max_delta_step'],
        'reg_alpha':best_param['reg_alpha'],
        'reg_lambda':best_param['reg_lambda'],
        'bagging_seed':101,
        'feature_fraction_seed':101,
        'random_state':101,
        'extra_seed':101,
}

final_pipeline = rf_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(
        col=mean_freq,
        name_= ['_mean','_frequency']
    )),
    ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("Light_GBM",LGBMRegressor(**lightgbm_params))
]).fit(X_train,y_train)

y_pred = final_pipeline.predict(X_test)

print(f"Mean Absolute Error: {mean_absolute_error(y_pred,y_test)}")

In [ ]:
final_mae = mean_absolute_error(y_test,y_pred)
final_mape = mean_absolute_percentage_error(y_test,y_pred)

print(f"Final Model MAE: {np.round(final_mae,2)}")
print(f"Final Model MAPE: {np.round((final_mape*100),2)}%")

The model was approximately 1.13% off on average when predicting unseen lap times in the test set. If future deployment data are similar to the training and test data, we can expect the model’s predictions to be off by approximately 1.13% on average.

In [ ]:
df_pred_overview = X_test.copy()
df_pred_overview['y_pred'] = y_pred

df_pred_overview['actual'] = y_test
df_pred_overview['pred_actual_diff'] = df_pred_overview['y_pred'] - df_pred_overview['actual']

bins = [-np.inf,-2,-1,-0.75,-0.5,-0.25,-0.1,0.0,0.1,0.25,0.5,0.75,1.0,2.0,np.inf]
df_pred_overview['diff_bins'] = pd.cut(df_pred_overview['pred_actual_diff'],bins=bins)

gp_input = 'Brazilian'
df_pred_diff_overview = df_pred_overview[df_pred_overview['GP'].eq(f'{gp_input} GP')][['Driver','Team','GP','actual','y_pred','pred_actual_diff','diff_bins']].sort_values('y_pred')

y_pred_rank_df = df_pred_diff_overview[['Driver','Team','GP','y_pred']].sort_values('y_pred').copy()
y_pred_rank_df['pred_rank'] = range(1,len(df_pred_diff_overview['Driver'])+1)
y_pred_rank_df.drop(['y_pred'],axis=1,inplace=True)
df_pred_diff_overview = df_pred_diff_overview.merge(y_pred_rank_df,how='inner',on=['Driver','Team','GP'])

actual_rank_df = df_pred_diff_overview[['Driver','Team','GP','actual']].sort_values('actual').copy()
actual_rank_df['actual_rank'] = range(1,len(df_pred_diff_overview['Driver'])+1)
actual_rank_df.drop(['actual'],axis=1,inplace=True)
df_pred_diff_overview = df_pred_diff_overview.merge(actual_rank_df,how='inner',on=['Driver','Team','GP'])
#postive value - we over-predicted the driver; negative value - we under-predicted the driver
df_pred_diff_overview['position_diff'] = df_pred_diff_overview['actual_rank'] - df_pred_diff_overview['pred_rank']
df_pred_diff_overview['position_diff_inter'] = df_pred_diff_overview['position_diff'].apply(lambda x: f'{x} over' if x > 0 else (f'{x*-1} under') if x < 0 else 'point on')

print(f"Avg. difference in position prediction: {df_pred_diff_overview['position_diff'].abs().mean()} & Standard Deviation for difference in position prediction: {df_pred_diff_overview['position_diff'].std()}")
print(f"MAE in predicted difference: {df_pred_diff_overview['pred_actual_diff'].abs().mean()}")
df_pred_diff_overview.drop(['position_diff'],axis=1)

In [ ]:
sns.histplot(data = df_pred_overview,x = 'pred_actual_diff')

In [ ]:
bins_vis = df_pred_overview['diff_bins'].value_counts()

bins_vis.plot(kind='bar',figsize=(12,6))
plt.title('Barplot of over-/under-prediction bins',size=16,fontweight='bold')

# Offset Approach

Next, we will create the offset for the 2026 predictions. Why do we do that? In 2026, there was a rule change, which made the car slower to address that, we will use an offset. This offset will be be created as following

1. We will fit the whole training data to the model
2. Then we will use five GPs of the 2026 season to create the offset
    - The offsets are the residuals of actual and predictions for the 2026 data
    - Driver offset (mean out of the five GPs, based on driver)
    - Team offset (mean out of the five GPs, based on team)
    - GP offset (mean out of the five GPs, based on GP)
3. Then I make predictions and add the average out of Driver, Team and GP offset to the prediction

In [ ]:
df_2026 = df[df['Season'].eq(2026)]
df_2026.head()

In [ ]:
#here we make sure that we only use a portion of the hold out to get the offsets
#the rest will be used to to see if the off set helps to improve
offset_gp_id = np.sort(df_2026['gp_id'].unique())[:5]
test_offset = np.sort(df_2026['gp_id'].unique())[5:]

offset_train_df = df[df['gp_id'].isin(offset_gp_id)]
off_train_X_train = offset_train_df.drop(['Session','LapTimeDiff_quali','laptime_sum_sectortimes_quali'],axis=1)
off_train_y_train = offset_train_df['laptime_sum_sectortimes_quali']

offset_test_df = df[df['gp_id'].isin(test_offset)]
off_train_X_test = offset_test_df.drop(['Session','LapTimeDiff_quali','laptime_sum_sectortimes_quali'],axis=1)
off_train_y_test = offset_test_df['laptime_sum_sectortimes_quali']

X = pd.concat([X_train,X_test],axis=0)
X.to_csv('../data/model_data/final/final_X.csv',index=False)
y = pd.concat([y_train,y_test],axis=0)
y.to_csv('../data/model_data/final/final_y.csv',index=False)

In [ ]:
best_param = joblib.load(r'..\model\params\lightgbm_best_param.joblib')

#list with all p2 and p3 features, which will be NaN, when we have a Sprint-Weekend
missing_in_sprint_we = [sprint_not for sprint_not in X_train.columns if 'p2' in sprint_not or 'p3' in sprint_not]
#
#mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
#'Team','Team_Lineage','Complexity_label','Session','Sprint-Session','Sprint_Race_Era']
mean_freq = ['Driver','GP','Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali',
'Team','Team_Lineage','Complexity_label','Sprint-Session','Sprint_Race_Era']
mean_enc_cols = [m+'_mean' for m in mean_freq]
freq_enc_cols = [f+'_frequency' for f in mean_freq]

lightgbm_params = {
        'objective':best_param['objective'],
        'max_bin':best_param['max_bin'],
        'num_iterations':best_param['num_iterations'],
        'learning_rate':best_param['learning_rate'],
        'num_leaves':best_param['num_leaves'],
        'num_threads':-1,
        'max_depth':best_param['max_depth'],
        'min_data_in_leaf':best_param['min_samples_leaf'],
        'min_sum_hessian_in_leaf':best_param['min_sum_hessian_in_leaf'],
        'bagging_fraction': best_param['bagging_fraction'],
        'bagging_freq': best_param['bagging_freq'],
        'colsample_bytree':best_param['colsample_bytree'],
        'feature_fraction_bynode':best_param['feature_fraction_bynode'],
        'extra_trees':best_param['extra_trees'],
        'max_delta_step':best_param['max_delta_step'],
        'reg_alpha':best_param['reg_alpha'],
        'reg_lambda':best_param['reg_lambda'],
        'bagging_seed':101,
        'feature_fraction_seed':101,
        'random_state':101,
        'extra_seed':101,
}

final_pipeline = rf_pipeline = Pipeline([
    ("RareLabel_Driver",RareLabelEncoder(variables='Driver',tol=best_param['tol_rare_label_driver'],replace_with='Rare_Driver')),
    ("RareLabelEncoder_GP",RareLabelEncoder(variables='GP',tol=best_param['tol_rare_label_gp'],replace_with='Rare_GP')),
    #due to sprint weekends, some rows for the practice 2 and 3 compound tyre features include NaN, because we set missing_values='ignore'
    #because tree-based models can deal with NaN values
    ("RareLabelEncoder_Compounds",RareLabelEncoder(
        variables=['Compound_p1','Compound_p2','Compound_p3','Compound_sprint_quali','Compound_quali'],
        tol=best_param['tol_rare_label_compounds'],
        n_categories=2,
        replace_with='Rare_Compound',
        missing_values='ignore'
    )),
    ('Adding_Columns_for_mean_&_frequency',Add_Column(
        col=mean_freq,
        name_= ['_mean','_frequency']
    )),
    ("Indicating_Missing_Value",AddMissingIndicator(missing_only=True,variables=missing_in_sprint_we)),
    ("Drop_Features",DropFeatures(features_to_drop=mean_freq)),
    ('One_Hot_Encoded_Features',OneHotEncoder(variables=['Pace_profile','Type','Direction','Rule_Era'])),
    ("Mean_encoding",MeanEncoder(variables=mean_enc_cols,missing_values='ignore',smoothing=best_param['mean_smoothing'],unseen="encode")),
    ("Frequency_encoding",CountFrequencyEncoder(variables=freq_enc_cols,missing_values='ignore',encoding_method='frequency',unseen="encode")),
    ("Feature_Selector",RFPermutationRegressorSelector(threshold=best_param['feature_selector'])),
    ("Light_GBM",LGBMRegressor(**lightgbm_params))
]).fit(X,y)

joblib.dump(final_pipeline,'../model/final_model/final_model_qualifying.joblib')

In [ ]:
off_train_X_train['GP'].unique()

In [ ]:
pred_offset_df = off_train_X_train.copy()

off_train_y_pred = final_pipeline.predict(off_train_X_train)
pred_offset_df['pred'] = off_train_y_pred
pred_offset_df['actual'] = off_train_y_train
pred_offset_df['resid'] = pred_offset_df['actual'] - pred_offset_df['pred'] 
pred_offset_df[pred_offset_df['GP'].eq('Miami GP')][['Driver','Team','GP','pred','actual','resid']].sort_values('pred')

In [ ]:
#driver offset
driver_offset = pred_offset_df.groupby(['Driver'],as_index=False)['resid'].median().rename(columns={'resid':'driver_offset'})
driver_offset.to_csv('../model/offset/driver_offset.csv',index=False)

#team offset
team_offset = pred_offset_df.groupby(['Team'],as_index=False)['resid'].median().rename(columns={'resid':'team_offset'})
team_offset.to_csv('../model/offset/team_offset.csv',index=False)

#GP offset
gp_offset = pred_offset_df.groupby(['GP'],as_index=False)['resid'].median().rename(columns={'resid':'gp_offset'})
global_gp_offset = gp_offset['gp_offset'].median()
joblib.dump(global_gp_offset,'../model/offset/global_gp_offset.joblib')

In [ ]:
pred_offset_add_df = off_train_X_test.copy()
off_test_y_pred = final_pipeline.predict(off_train_X_test)
pred_offset_add_df['pred'] = off_test_y_pred
pred_offset_add_df['actual'] = off_train_y_test
pred_offset_add_df['resid'] = pred_offset_add_df['actual'] - pred_offset_add_df['pred']

pred_offset_add_df = pd.merge(pred_offset_add_df,driver_offset,how='inner',on='Driver')
pred_offset_add_df = pd.merge(pred_offset_add_df,team_offset,how='inner',on='Team')
pred_offset_add_df['gp_global_offset'] = global_gp_offset
pred_offset_add_df['offset'] = pred_offset_add_df[['driver_offset','team_offset','gp_global_offset']].mean(axis=1)

def def_apply_offset_mean(
        df=df,
        pred='pred',
        offset='offset',
        shrinking=1.0,
        limit=None
):
    result = df.copy()
    correction = result[offset].fillna(0.0)*shrinking

    #here we apply a limit, the limit will be applied when the value is not None
    if limit is not None:
        correction = np.clip(
            lower = -limit,
            upper = limit
        )
    result['applied_offset'] = correction
    result['pred_corrected'] = result[pred] + result['applied_offset']
    return result

pred_offset_add_df = def_apply_offset_mean(df=pred_offset_add_df,shrinking=0.55)
pred_offset_add_df['offset_resid'] = pred_offset_add_df['actual'] - pred_offset_add_df['pred_corrected']
pred_offset_add_df['improve_status'] = np.where(pred_offset_add_df['offset_resid'].abs()<pred_offset_add_df['resid'].abs(),'Improved','Worse')
pred_offset_add_df[['Driver','Team','GP','actual','pred','resid','pred_corrected','offset_resid','improve_status']].head()

In [ ]:
mean_absolute_error(pred_offset_add_df['actual'],pred_offset_add_df['pred'])

In [ ]:
mean_absolute_error(pred_offset_add_df['actual'],pred_offset_add_df['pred_corrected'])

In [ ]:
def tune_offset_shrinkage_expanding(
    calibration_df,
    shrinking_values=None,
    gp_col="gp_id",
    driver_col="Driver",
    team_col="Team",
    residual_col="resid",
    prediction_col="pred",
    actual_col="actual",
):
    """
    Tune one shrinking value using expanding-window validation.

    GP2 uses GP1 offsets.
    GP3 uses GP1-GP2 offsets.
    ...
    """

    #if we do not provide a shrinking range, it will us the default range
    if shrinking_values is None:
        shrinking_values = np.arange(0.0, 1.01, 0.05)

    #here we will get the unique values for all avaiable gp_ids in the dataframe
    gp_ids = np.sort(calibration_df[gp_col].unique())
    results = []

    #here we loop different shrinking_values
    for shrinking in shrinking_values:
        expanding_predictions = []

        #start with GP2 because GP1 has no previous 2026 information
        #to avoid using GP1 in the validation, we start the range with 1
        for i in range(1, len(gp_ids)):
            #making sure to include everything up to the range value
            previous_gp_ids = gp_ids[:i]
            current_gp_id = gp_ids[i]

            history = calibration_df[
                calibration_df[gp_col].isin(previous_gp_ids)
            ]

            current_gp = calibration_df[
                calibration_df[gp_col].eq(current_gp_id)
            ].copy()

            global_offset = history[residual_col].median()

            #creating the different offsets
            driver_offsets = (
                history.groupby(driver_col)[residual_col].median()
            )

            team_offsets = (
                history.groupby(team_col)[residual_col].median()
            )

            current_gp["driver_offset"] = (
                current_gp[driver_col]
                .map(driver_offsets)
                .fillna(global_offset)
            )

            current_gp["team_offset"] = (
                current_gp[team_col]
                .map(team_offsets)
                .fillna(global_offset)
            )

            #adding the offsets to the current GP in the expanding window
            current_gp["gp_global_offset"] = global_offset

            #taking the mean to get my offset, that will be applied to the prediction
            current_gp["offset"] = current_gp[
                [
                    "driver_offset",
                    "team_offset",
                    "gp_global_offset",
                ]
            ].mean(axis=1)

            current_gp["pred_corrected"] = (
                current_gp[prediction_col]
                + shrinking * current_gp["offset"]
            )

            expanding_predictions.append(current_gp)

        expanding_df = pd.concat(
            expanding_predictions,
            ignore_index=True,
        )

        mae = mean_absolute_error(
            expanding_df[actual_col],
            expanding_df["pred_corrected"],
        )

        results.append(
            {
                "shrinking": shrinking,
                "mae": mae,
            }
        )

    results_df = (
        pd.DataFrame(results)
        .sort_values("mae")
        .reset_index(drop=True)
    )

    best_shrinking = results_df.loc[0, "shrinking"]

    return best_shrinking, results_df

In [ ]:
best_shrinking, shrinking_results = (
    tune_offset_shrinkage_expanding(
        calibration_df=pred_offset_df,
        shrinking_values=np.arange(0.0, 1.01, 0.05),
    )
)

print(f"Best shrinking: {best_shrinking:.2f}")
print(shrinking_results.head(100))

In [ ]:
best_shrinking

In [ ]:
global_gp_offset = pred_offset_df["resid"].mean()

driver_offset = (
    pred_offset_df
    .groupby("Driver", as_index=False)["resid"]
    .mean()
    .rename(columns={"resid": "driver_offset"})
)

team_offset = (
    pred_offset_df
    .groupby("Team", as_index=False)["resid"]
    .mean()
    .rename(columns={"resid": "team_offset"})
)

pred_offset_add_df = off_train_X_test.copy()

pred_offset_add_df["pred"] = final_pipeline.predict(
    off_train_X_test
)

pred_offset_add_df["actual"] = off_train_y_test.to_numpy()

pred_offset_add_df = pred_offset_add_df.merge(
    driver_offset,
    on="Driver",
    how="left",
)

pred_offset_add_df = pred_offset_add_df.merge(
    team_offset,
    on="Team",
    how="left",
)

# Unseen drivers or teams fall back to the global correction
pred_offset_add_df["driver_offset"] = (
    pred_offset_add_df["driver_offset"]
    .fillna(global_gp_offset)
)

pred_offset_add_df["team_offset"] = (
    pred_offset_add_df["team_offset"]
    .fillna(global_gp_offset)
)

pred_offset_add_df["gp_global_offset"] = global_gp_offset

pred_offset_add_df["offset"] = pred_offset_add_df[
    [
        "driver_offset",
        "team_offset",
        "gp_global_offset",
    ]
].mean(axis=1)

pred_offset_add_df["applied_offset"] = (
    best_shrinking * pred_offset_add_df["offset"]
)

pred_offset_add_df["pred_corrected"] = (
    pred_offset_add_df["pred"]
    + pred_offset_add_df["applied_offset"]
)

In [ ]:
pred_offset_add_df[['Driver','Team','GP','actual','pred','offset','pred_corrected','applied_offset']]

In [ ]:
final_model = joblib.load('../model/final_model/final_model_qualifying.joblib')
df_offset_pred = off_train_X_test.copy()


y_pred_offset = final_model.predict(off_train_X_test)
df_offset_pred['Pred_time'] = y_pred_offset

#driver offset
driver_offset = pd.read_csv('../model/offset/driver_offset.csv').set_index("Driver")["driver_offset"]
#team offset
team_offset = pd.read_csv('../model/offset/team_offset.csv').set_index("Team")["team_offset"]
#GP offset
global_gp_offset = joblib.load('../model/offset/global_gp_offset.joblib')


df_offset_pred["driver_offset"] = (df_offset_pred["Driver"].map(driver_offset).fillna(global_gp_offset))
df_offset_pred["team_offset"] = (df_offset_pred["Team"].map(team_offset).fillna(global_gp_offset))
df_offset_pred['gp_global_offset'] = global_gp_offset
df_offset_pred['offset'] = df_offset_pred[['driver_offset','team_offset','gp_global_offset']].mean(axis=1)
df_offset_pred = def_apply_offset_mean(df=df_offset_pred,pred='Pred_time',shrinking=0.55)

final_mae = mean_absolute_error(off_train_y_test,df_offset_pred['pred_corrected'])
final_mape = mean_absolute_percentage_error(off_train_y_test,df_offset_pred['pred_corrected'])

print(f"Final Model MAE: {np.round(final_mae,2)}")
print(f"Final Model MAPE: {np.round((final_mape*100),2)}%")

# Creating Input Values

Here we have two options:
- We manually input the values
- We use CSV files for existing data of the weekend

Why should we use teh CSV file approach? The model uses the fastest laps from P1,P2,P3 and Sprint-Race pole, so we need data from the weekend, to be more accurate. This model should be used before the qualifying.

## Manual Input

In [ ]:
df = pd.read_csv(r'..\data\processed\Cleaned_quali_f1.csv')
df_circut = pd.read_excel(r'..\data\raw\f1_circuits_2018_2026_extended.xlsx')
df_cols = df.columns

df_cols = [d for d in df.columns if d not in ['Session','laptime_sum_sectortimes_quali','LapTimeDiff_quali']]
input_request = {d:f'Please enter the {d}' for d in df_cols}

input_values = {}

for feat in df_cols:
    value = input(input_request[feat])
    #how to check for a pandas series the data type
    input_values[feat] = df[feat].dytpe.type(value)

input_df = pd.DataFrame([input_values])

## CSV File Approach

In [ ]:
#setting up the F1 season claendar
f1_calendar_by_season = {
    2018: [
        "Australian GP", "Bahrain GP", "Chinese GP", "Azerbaijan GP",
        "Spanish GP", "Monaco GP", "Canadian GP", "French GP",
        "Austrian GP", "British GP", "German GP", "Hungarian GP",
        "Belgian GP", "Italian GP", "Singapore GP", "Russian GP",
        "Japanese GP", "United States GP", "Mexican GP", "Brazilian GP",
        "Abu Dhabi GP",
    ],

    2019: [
        "Australian GP", "Bahrain GP", "Chinese GP", "Azerbaijan GP",
        "Spanish GP", "Monaco GP", "Canadian GP", "French GP",
        "Austrian GP", "British GP", "German GP", "Hungarian GP",
        "Belgian GP", "Italian GP", "Singapore GP", "Russian GP",
        "Japanese GP", "Mexican GP", "United States GP", "Brazilian GP",
        "Abu Dhabi GP",
    ],

    2020: [
        "Austrian GP", "Styrian GP", "Hungarian GP", "British GP",
        "70th Anniversary GP", "Spanish GP", "Belgian GP", "Italian GP",
        "Tuscan GP", "Russian GP", "Eifel GP", "Portuguese GP",
        "Emilia Romagna GP", "Turkish GP", "Bahrain GP", "Sakhir GP",
        "Abu Dhabi GP",
    ],

    2021: [
        "Bahrain GP", "Emilia Romagna GP", "Portuguese GP", "Spanish GP",
        "Monaco GP", "Azerbaijan GP", "French GP", "Styrian GP",
        "Austrian GP", "British GP", "Hungarian GP", "Belgian GP",
        "Dutch GP", "Italian GP", "Russian GP", "Turkish GP",
        "United States GP", "Mexican GP", "Brazilian GP", "Qatar GP",
        "Saudi Arabian GP", "Abu Dhabi GP",
    ],

    2022: [
        "Bahrain GP", "Saudi Arabian GP", "Australian GP", "Emilia Romagna GP",
        "Miami GP", "Spanish GP", "Monaco GP", "Azerbaijan GP",
        "Canadian GP", "British GP", "Austrian GP", "French GP",
        "Hungarian GP", "Belgian GP", "Dutch GP", "Italian GP",
        "Singapore GP", "Japanese GP", "United States GP", "Mexican GP",
        "Brazilian GP", "Abu Dhabi GP",
    ],

    2023: [
        "Bahrain GP", "Saudi Arabian GP", "Australian GP", "Azerbaijan GP",
        "Miami GP", "Monaco GP", "Spanish GP", "Canadian GP",
        "Austrian GP", "British GP", "Hungarian GP", "Belgian GP",
        "Dutch GP", "Italian GP", "Singapore GP", "Japanese GP",
        "Qatar GP", "United States GP", "Mexican GP", "Brazilian GP",
        "Las Vegas GP", "Abu Dhabi GP",
    ],

    2024: [
        "Bahrain GP", "Saudi Arabian GP", "Australian GP", "Japanese GP",
        "Chinese GP", "Miami GP", "Emilia Romagna GP", "Monaco GP",
        "Canadian GP", "Spanish GP", "Austrian GP", "British GP",
        "Hungarian GP", "Belgian GP", "Dutch GP", "Italian GP",
        "Azerbaijan GP", "Singapore GP", "United States GP", "Mexican GP",
        "Brazilian GP", "Las Vegas GP", "Qatar GP", "Abu Dhabi GP",
    ],

    2025: [
        "Australian GP", "Chinese GP", "Japanese GP", "Bahrain GP",
        "Saudi Arabian GP", "Miami GP", "Emilia Romagna GP", "Monaco GP",
        "Spanish GP", "Canadian GP", "Austrian GP", "British GP",
        "Belgian GP", "Hungarian GP", "Dutch GP", "Italian GP",
        "Azerbaijan GP", "Singapore GP", "United States GP", "Mexican GP",
        "Brazilian GP", "Las Vegas GP", "Qatar GP", "Abu Dhabi GP",
    ],

    #2026 was already adjusted due to the canceld Saudi Arabia and Barhain GPs
    2026: [
        "Australian GP", "Chinese GP", "Japanese GP", "Miami GP",
        "Canadian GP", "Monaco GP", "Spanish GP", "Austrian GP",
        "British GP", "Belgian GP", "Hungarian GP", "Dutch GP",
        "Italian GP", "Madrid GP", "Azerbaijan GP", "Singapore GP",
        "United States GP", "Mexican GP", "Brazilian GP", "Las Vegas GP",
        "Qatar GP", "Abu Dhabi GP",
    ],
}

#setting up roots
#in a notebook not needed, but we will need it later in the Python file anyways
ROOT = Path.cwd().parent
DATA = ROOT /"data"
DATA_RAW = DATA / "raw"
DATA_OUTPUT = DATA/"processed"

#loading the circut information data frame
df_circut = pd.read_csv(DATA_OUTPUT / 'cleaned_f1_circut.csv')

df_2018_2026 = pd.DataFrame()
start_range = 2018
end_range = 2026

for season in range(start_range,end_range+1):
    #turning "season" to string, because the path cannot processes a numerical value
    data_path = DATA_RAW / str(season)

    #getting name of folder if the folder exists
    path_list = sorted([f.name for f in data_path.iterdir() if f.is_dir() and not f.name.startswith('.') and f.name.endswith("GP")])

    # keep this only if you really need it
    #if season <= 2026:
    #path_list = path_list[1:]

    if season == 2020:
        path_list = [x for x in path_list if x != "Eifel GP"]

    #entering each folder of a Grand Prix weekend
    for folder_loop_value in path_list:
        base = data_path / folder_loop_value

        timing_paths, weather_paths = build_session_paths(
            base=base,
            season=season,
            folder_loop_value=folder_loop_value
        )

        df_loop = practice_quali_new_pred(
            season=season,
            gp=folder_loop_value,
            timing_paths=timing_paths,
            weather_paths=weather_paths,
            w_vis=False
        )

        df_2018_2026 = pd.concat([df_2018_2026, df_loop], ignore_index=True)
#creating the mapping => key: season, gp; value: number of the gp-weekend
race_round_map = {
    (season, gp): round_no
    for season, races in f1_calendar_by_season.items()
    for round_no, gp in enumerate(races, start=1)
}

#race_round will be used as an X-feature
df_2018_2026["race_round"] = df_2018_2026.apply(
    lambda row: race_round_map.get((row["Season"], row["GP"])),
    axis=1
)

#creating a dataframe to create an race order
weekend_order = (
    df_2018_2026[["Season", "GP", "race_round"]]
    .drop_duplicates()
    .sort_values(["Season", "race_round"])
    .reset_index(drop=True)
)
#creating a unique id based on the race weekend order
#group_id will be used for time-aware cross-validation
weekend_order["gp_id"] = range(len(weekend_order))

#finally joining the race order logic
df_2018_2026 = df_2018_2026.merge(
    weekend_order[["Season", "GP", "gp_id"]],
    on=["Season", "GP"],
    how="left"
)

team_lineage_map = {
    "Toro Rosso": "Toro Rosso-AlphaTauri-RB-Racing Bulls",
    "AlphaTauri": "Toro Rosso-AlphaTauri-RB-Racing Bulls",
    "RB": "Toro Rosso-AlphaTauri-RB-Racing Bulls",
    "Racing Bulls": "Toro Rosso-AlphaTauri-RB-Racing Bulls",

    "Sauber": "Sauber-Alfa Romeo-Kick Sauber-Audi",
    "Alfa Romeo": "Sauber-Alfa Romeo-Kick Sauber-Audi",
    "Alfa Romeo Racing": "Sauber-Alfa Romeo-Kick Sauber-Audi",
    "Kick Sauber": "Sauber-Alfa Romeo-Kick Sauber-Audi",
    "Audi": "Sauber-Alfa Romeo-Kick Sauber-Audi",

    "Force India":"Force India-Racing Point-Aston Martin",
    "Racing Point":"Force India-Racing Point-Aston Martin",
    "Aston Martin":"Force India-Racing Point-Aston Martin",
    "Alpine":"Alpine-Renault",
    "Renault":"Alpine-Renault"
}

df_2018_2026["Team_Lineage"] = df_2018_2026["Team"].replace(team_lineage_map)
df_quali = pd.merge(df_2018_2026,df_circut,how='left',on='GP')
df_quali["Pace_profile"] = df_quali["Pace_profile"].str.strip()

#determining the Grand Prix
gp_f = str(input('Please enter the GP that you want to predict'))
#using the most recent data for teh Grand Prix
current_season = df_quali['Season'].max()
df_quali_pred = df_quali[(df_quali['GP'].eq(f'{gp_f} GP')) & (df_quali['Season'].eq(current_season))]

df_quali_pred = df_quali_pred.drop(['TrackTemp_p1','TrackTemp_p2','TrackTemp_p3','TrackTemp_sprint_quali'],axis=1)

add_feat = [d for d in df.columns if 'sprint' not in d and ('quali' in d and d not in ['LapTimeDiff_quali','laptime_sum_sectortimes_quali'])]
null_check_df = df_quali_pred.isnull().sum().to_frame()
null_lst = set(n for n in null_check_df[null_check_df[0]>10].index)
total_set_check = set(d for d in df_quali_pred.columns if d.startswith(('AirTemp_','Humidity_','Pressure_','Rainfall_')))
mean_valid_cols = total_set_check - null_lst

for a in add_feat:
    sub_mean_cols = [smc for smc in mean_valid_cols if smc.split('_')[0] == a.split('_')[0]]
    print(sub_mean_cols)
    input_add_feat = float(np.mean([df_quali_pred[x].mean() for x in sub_mean_cols]))
    df_quali_pred[a]= input_add_feat
df_quali_pred['Compound_quali'] = input("Please enter Compound: ").strip().upper()
df_quali_pred = df_quali_pred[[x for x in X_train.columns]].copy()

#final_model = joblib.load('../model/final_model/final_model_qualifying.joblib')

current = Path.cwd()

PROJECT_ROOT = None

for folder in [current, *current.parents]:
    if (folder / "src" / "py_def_class.py").exists():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root.")

# Make `src` importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Optional sanity check
import src.py_def_class

final_model = joblib.load(
    PROJECT_ROOT
    / "model"
    / "final_model"
    / "final_model_qualifying.joblib"
)

quali_pred_values = final_model.predict(df_quali_pred)
df_quali_pred['Pred_time'] = quali_pred_values

#driver offset
driver_offset = pd.read_csv('../model/offset/driver_offset.csv').set_index("Driver")["driver_offset"]
#team offset
team_offset = pd.read_csv('../model/offset/team_offset.csv').set_index("Team")["team_offset"]
#GP offset
global_gp_offset = joblib.load('../model/offset/global_gp_offset.joblib')


df_quali_pred["driver_offset"] = (df_quali_pred["Driver"].map(driver_offset).fillna(global_gp_offset))
df_quali_pred["team_offset"] = (df_quali_pred["Team"].map(team_offset).fillna(global_gp_offset))
df_quali_pred['gp_global_offset'] = global_gp_offset
df_quali_pred['offset'] = df_quali_pred[['driver_offset','team_offset','gp_global_offset']].mean(axis=1)
df_quali_pred = def_apply_offset_mean(df=df_quali_pred,pred='Pred_time',shrinking=0.55)
df_quali_pred[['Driver','Team','GP','pred_corrected']]

In [ ]:
df_quali_pred.head()